# 03 — Air4Thai Station Metadata, Thailand Station Map, and CSV Export

**ระดับ:** ปริญญาโท / ผู้เริ่มต้น–ระดับกลาง  
**แพลตฟอร์ม:** Google Colab  
**ข้อมูล:** Air4Thai / Pollution Control Department (PCD), Thailand

## วัตถุประสงค์
Notebook นี้สอนให้นิสิตสามารถ

1. ดาวน์โหลด **station metadata** จาก Air4Thai
2. ตรวจสอบ `stationID`, ชื่อสถานี, พื้นที่, latitude และ longitude
3. ตรวจคุณภาพข้อมูลพิกัดและสถานีซ้ำ
4. สร้างตาราง metadata ของทุกสถานี
5. สร้าง GeoDataFrame สำหรับงาน GIS
6. พลอตตำแหน่งสถานีทั้งหมดบนแผนที่ประเทศไทย
7. สร้าง interactive map ด้วย Folium
8. export ข้อมูลสถานีเป็น CSV สำหรับนำไป join กับข้อมูล PM2.5 รายวัน
9. export GeoJSON สำหรับใช้ใน QGIS/ArcGIS (optional)

> Workflow: **Air4Thai API → JSON → Pandas → QC → GeoDataFrame → Thailand Map → CSV/GeoJSON**

## 0. หมายเหตุสำคัญ

- Air4Thai API อาจปรับโครงสร้างข้อมูลได้ในอนาคต ดังนั้น Notebook นี้ใช้วิธีอ่าน JSON แบบค่อนข้างยืดหยุ่น
- พิกัดที่ได้จาก API ควรถือเป็น **current station metadata** และควรตรวจสอบประวัติการย้ายสถานี หากนำไปใช้ในงานวิจัยย้อนหลังหลายปี
- สำหรับงานวิจัย 5–10 ปี ควรบันทึกไฟล์ metadata พร้อมวันที่ดาวน์โหลดเพื่อ reproducibility

## 1. ติดตั้งไลบรารี

In [ ]:
!pip -q install geopandas folium requests

## 2. Import libraries

In [ ]:
import json
import time
from pathlib import Path
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import folium
from folium.plugins import MarkerCluster

print("Libraries imported successfully.")

## 3. Mount Google Drive และสร้างโฟลเดอร์ผลลัพธ์

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/AirPollution_Course")
OUT_DIR = BASE_DIR / "03_air4thai_station_metadata"
FIG_DIR = OUT_DIR / "figures"
DATA_DIR = OUT_DIR / "data"

for folder in [OUT_DIR, FIG_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)

## 4. กำหนด Air4Thai API

Endpoint ที่ใช้สำหรับ station metadata และข้อมูลคุณภาพอากาศล่าสุด:

In [ ]:
AIR4THAI_URL = "https://air4thai.pcd.go.th/services/getNewAQI_JSON.php"

print(AIR4THAI_URL)

## 5. ดาวน์โหลด JSON จาก Air4Thai

เราใช้ `requests.Session()` และ retry เบื้องต้น เพื่อให้ทนต่อปัญหา network ชั่วคราว

> **หมายเหตุ (fix SSL):** เซิร์ฟเวอร์ `air4thai.pcd.go.th` มักส่ง **certificate chain ไม่ครบ**
> (ขาด intermediate CA) ทำให้ Python ตรวจใบรับรองไม่ผ่าน แม้เว็บจะเปิดได้ปกติในเบราว์เซอร์
> จะเห็น error ว่า `CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate`
> การอัปเดต `certifi` **ไม่ช่วย** เพราะปัญหาอยู่ที่ฝั่งเซิร์ฟเวอร์ ไม่ใช่ที่เครื่องเรา
>
> ฟังก์ชันด้านล่างจึงทำงานแบบ 2 ชั้น: ลองแบบ **ตรวจใบรับรองตามปกติก่อน (ปลอดภัยสุด)**
> ถ้าเจอ SSL error จึงค่อยลองใหม่แบบ **ไม่ตรวจใบรับรอง (`verify=False`)** พร้อมแจ้งเตือน
> โหมด `verify=False` เหมาะกับ API สาธารณะแบบนี้ แต่ไม่ควรใช้กับข้อมูลที่ต้องรักษาความลับ

In [ ]:
import urllib3


def download_air4thai_json(url, max_retries=3, timeout=60):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 Chrome/120 Safari/537.36"
        ),
        "Accept": "application/json,text/plain,*/*",
    }

    session = requests.Session()
    last_error = None

    # เริ่มด้วยการตรวจใบรับรอง SSL ตามปกติ (ปลอดภัยที่สุด)
    verify_ssl = True

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries} (verify_ssl={verify_ssl}) ...")

            if not verify_ssl:
                # ปิด warning ของ urllib3 เมื่อจงใจไม่ตรวจใบรับรอง
                urllib3.disable_warnings(
                    urllib3.exceptions.InsecureRequestWarning
                )

            response = session.get(
                url,
                headers=headers,
                timeout=timeout,
                verify=verify_ssl,
            )
            response.raise_for_status()

            data = response.json()
            print("Download successful.")
            return data

        except requests.exceptions.SSLError as e:
            last_error = e
            print("SSL verification failed:", e)

            # air4thai.pcd.go.th มักส่ง certificate chain ไม่ครบ
            # ครั้งแรกที่เจอ SSL error ให้สลับไปโหมดไม่ตรวจใบรับรองแล้วลองใหม่ทันที
            if verify_ssl:
                print(">> เปลี่ยนเป็นโหมดไม่ตรวจ SSL (verify=False) แล้วลองใหม่ ...")
                verify_ssl = False
                continue

            if attempt < max_retries:
                time.sleep(3)

        except Exception as e:
            last_error = e
            print("Download failed:", e)

            if attempt < max_retries:
                time.sleep(3)

    raise RuntimeError(
        f"Unable to download Air4Thai data after {max_retries} attempts."
    ) from last_error


air4thai_json = download_air4thai_json(AIR4THAI_URL)

## 6. สำรวจโครงสร้าง JSON ก่อนวิเคราะห์

นี่เป็นนิสัยสำคัญของการทำงานกับ API: **อย่าเดาโครงสร้างข้อมูล**

In [ ]:
print("Python object type:", type(air4thai_json))

if isinstance(air4thai_json, dict):
    print("\nTop-level keys:")
    print(list(air4thai_json.keys()))

    for key, value in air4thai_json.items():
        print(f"{key}: {type(value)}")

elif isinstance(air4thai_json, list):
    print("\nNumber of top-level records:", len(air4thai_json))

## 7. ดึงรายการสถานีออกจาก JSON

Air4Thai มักเก็บ station list ภายใต้ key `stations`  
โค้ดนี้มี fallback หาก API เปลี่ยนรูปแบบเป็น list โดยตรง

In [ ]:
if isinstance(air4thai_json, dict) and "stations" in air4thai_json:
    stations = air4thai_json["stations"]

elif isinstance(air4thai_json, list):
    stations = air4thai_json

else:
    raise ValueError(
        "ไม่พบ station list ใน JSON กรุณาตรวจโครงสร้าง API ใน Cell ก่อนหน้า"
    )

print("Number of station records:", len(stations))

## 8. ดูข้อมูลของสถานีหนึ่งแห่ง

ใช้เพื่อเรียนรู้ว่า Air4Thai ให้ field อะไรบ้าง

In [ ]:
if len(stations) > 0:
    print(json.dumps(stations[0], ensure_ascii=False, indent=2))

## 9. แปลง JSON → Pandas DataFrame

`pd.json_normalize()` มีข้อดีคือสามารถ flatten nested JSON เช่นข้อมูล AQI ล่าสุดออกมาเป็นคอลัมน์ได้ด้วย

In [ ]:
stations_full = pd.json_normalize(stations, sep="_")

print("Shape:", stations_full.shape)
print("\nColumns:")
for c in stations_full.columns:
    print(" -", c)

display(stations_full.head())

## 10. สร้างชื่อคอลัมน์มาตรฐานสำหรับงานวิจัย

เราจะเก็บ **ทุกคอลัมน์เดิมจาก API** ไว้ใน `stations_full`  
จากนั้นสร้างคอลัมน์มาตรฐานเพิ่ม ได้แก่

- `station_code`
- `station_name_th`
- `station_name_en`
- `area_th`
- `area_en`
- `station_type`
- `latitude`
- `longitude`

เพื่อให้ Notebook อื่นเรียกใช้ได้ง่าย

In [ ]:
def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_station = first_existing_column(
    stations_full,
    ["stationID", "stationId", "station_id", "stationCode"]
)

col_name_th = first_existing_column(
    stations_full,
    ["nameTH", "stationNameTH", "name_th"]
)

col_name_en = first_existing_column(
    stations_full,
    ["nameEN", "stationNameEN", "name_en"]
)

col_area_th = first_existing_column(
    stations_full,
    ["areaTH", "area_th"]
)

col_area_en = first_existing_column(
    stations_full,
    ["areaEN", "area_en"]
)

col_type = first_existing_column(
    stations_full,
    ["stationType", "station_type"]
)

col_lat = first_existing_column(
    stations_full,
    ["lat", "latitude", "Latitude"]
)

col_lon = first_existing_column(
    stations_full,
    ["long", "lon", "longitude", "Longitude"]
)

print("Detected columns")
print("----------------")
print("station :", col_station)
print("nameTH  :", col_name_th)
print("nameEN  :", col_name_en)
print("areaTH  :", col_area_th)
print("areaEN  :", col_area_en)
print("type    :", col_type)
print("lat     :", col_lat)
print("lon     :", col_lon)

if col_station is None or col_lat is None or col_lon is None:
    raise ValueError(
        "ไม่พบ station ID หรือ latitude/longitude กรุณาตรวจชื่อคอลัมน์จาก API"
    )

In [ ]:
stations_full["station_code"] = (
    stations_full[col_station]
    .astype(str)
    .str.strip()
    .str.upper()
)

stations_full["station_name_th"] = (
    stations_full[col_name_th] if col_name_th else np.nan
)

stations_full["station_name_en"] = (
    stations_full[col_name_en] if col_name_en else np.nan
)

stations_full["area_th"] = (
    stations_full[col_area_th] if col_area_th else np.nan
)

stations_full["area_en"] = (
    stations_full[col_area_en] if col_area_en else np.nan
)

stations_full["station_type"] = (
    stations_full[col_type] if col_type else np.nan
)

stations_full["latitude"] = pd.to_numeric(
    stations_full[col_lat],
    errors="coerce"
)

stations_full["longitude"] = pd.to_numeric(
    stations_full[col_lon],
    errors="coerce"
)

display(
    stations_full[
        [
            "station_code",
            "station_name_th",
            "area_th",
            "station_type",
            "latitude",
            "longitude"
        ]
    ].head(10)
)

## 11. Quality Control ของ station metadata

ตรวจ 4 เรื่อง:

1. station code ซ้ำหรือไม่
2. latitude/longitude ขาดหรือไม่
3. พิกัดอยู่นอกช่วงทางคณิตศาสตร์หรือไม่
4. พิกัดอยู่ในบริเวณประเทศไทยโดยประมาณหรือไม่

In [ ]:
print("Total records:", len(stations_full))
print("Unique station codes:", stations_full["station_code"].nunique())

duplicate_codes = stations_full.loc[
    stations_full["station_code"].duplicated(keep=False),
    ["station_code", "station_name_th", "latitude", "longitude"]
].sort_values("station_code")

print("\nDuplicate station records:")
display(duplicate_codes)

missing_coord = stations_full[
    stations_full["latitude"].isna() |
    stations_full["longitude"].isna()
][
    ["station_code", "station_name_th", "area_th", "latitude", "longitude"]
]

print("\nStations with missing coordinates:")
display(missing_coord)

In [ ]:
# Mathematical coordinate validity
valid_math = (
    stations_full["latitude"].between(-90, 90) &
    stations_full["longitude"].between(-180, 180)
)

print(
    "Records with mathematically valid coordinates:",
    int(valid_math.sum()),
    "/",
    len(stations_full)
)

# Broad Thailand bounding box — ใช้เป็น QC flag เท่านั้น
# ไม่ใช้แทนขอบเขตประเทศจริง
in_thailand_bbox = (
    stations_full["latitude"].between(5.0, 21.5) &
    stations_full["longitude"].between(97.0, 106.5)
)

stations_full["coord_in_thailand_bbox"] = in_thailand_bbox

outside_bbox = stations_full.loc[
    valid_math & ~in_thailand_bbox,
    [
        "station_code",
        "station_name_th",
        "area_th",
        "latitude",
        "longitude"
    ]
]

print("\nValid coordinates outside broad Thailand QC box:")
display(outside_bbox)

## 12. สร้าง Core Station Metadata Table

ตารางนี้เหมาะสำหรับนำไป join กับ PM₂.₅ daily data เพราะไม่เก็บตัวแปร AQI ล่าสุดที่เปลี่ยนตามเวลา

In [ ]:
core_columns = [
    "station_code",
    "station_name_th",
    "station_name_en",
    "area_th",
    "area_en",
    "station_type",
    "latitude",
    "longitude",
    "coord_in_thailand_bbox"
]

station_core = (
    stations_full[core_columns]
    .drop_duplicates(subset="station_code")
    .sort_values("station_code")
    .reset_index(drop=True)
)

print("Unique stations:", len(station_core))
display(station_core.head(20))

## 13. บันทึกวันที่ดาวน์โหลด metadata

เพื่อ reproducibility เราจะเพิ่ม `metadata_retrieved_utc`

In [ ]:
retrieved_utc = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")

stations_full["metadata_retrieved_utc"] = retrieved_utc
station_core["metadata_retrieved_utc"] = retrieved_utc

print(retrieved_utc)

## 14. Export CSV

สร้าง 2 ไฟล์:

### Full metadata
เก็บทุก field ที่ Air4Thai ส่งมา รวมถึง nested/current AQI information ที่ `json_normalize()` flatten แล้ว

### Core metadata
เก็บเฉพาะข้อมูลสถานีที่เหมาะสำหรับ join กับฐานข้อมูลย้อนหลัง

In [ ]:
full_csv = DATA_DIR / "air4thai_station_metadata_FULL.csv"
core_csv = DATA_DIR / "air4thai_station_metadata_CORE.csv"

stations_full.to_csv(
    full_csv,
    index=False,
    encoding="utf-8-sig"
)

station_core.to_csv(
    core_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(full_csv)
print(core_csv)

## 15. สร้าง GeoDataFrame จาก Latitude / Longitude

Air4Thai coordinates เป็น geographic coordinates จึงกำหนด CRS เป็น **EPSG:4326 (WGS84)**

In [ ]:
station_map_df = station_core.dropna(
    subset=["latitude", "longitude"]
).copy()

station_gdf = gpd.GeoDataFrame(
    station_map_df,
    geometry=gpd.points_from_xy(
        station_map_df["longitude"],
        station_map_df["latitude"]
    ),
    crs="EPSG:4326"
)

station_gdf.head()

## 16. ดาวน์โหลดขอบเขตประเทศไทยจาก Natural Earth

ใช้ขอบเขตประเทศจริงสำหรับ cartographic context  
หาก URL เปลี่ยนในอนาคต สามารถแทนด้วย shapefile/GeoPackage ของประเทศไทยที่ผู้สอนเตรียมไว้

In [ ]:
NATURAL_EARTH_URL = (
    "https://raw.githubusercontent.com/nvkelso/"
    "natural-earth-vector/master/geojson/"
    "ne_50m_admin_0_countries.geojson"
)

# ดาวน์โหลดไฟล์ขอบเขตประเทศแล้ว cache ไว้ใน DATA_DIR
boundary_path = DATA_DIR / "ne_50m_admin_0_countries.geojson"


def download_boundary(url, dest, max_retries=3, timeout=120):
    # ถ้ามีไฟล์อยู่แล้ว ใช้ของเดิม (ไม่ต้องโหลดซ้ำ)
    if dest.exists() and dest.stat().st_size > 0:
        print("Using cached boundary file:", dest)
        return dest

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 Chrome/120 Safari/537.36"
        ),
        "Accept": "application/json,text/plain,*/*",
    }

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading Thailand boundary, attempt {attempt}/{max_retries} ...")
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            dest.write_bytes(r.content)
            print("Boundary download successful:", dest)
            return dest
        except Exception as e:
            last_error = e
            print("Download failed:", e)
            if attempt < max_retries:
                time.sleep(3)

    raise RuntimeError(
        "ไม่สามารถดาวน์โหลดขอบเขตประเทศไทยได้ "
        "หาก network มีปัญหา ให้อัปโหลด shapefile/GeoJSON เองแล้วตั้งค่า "
        "boundary_path ให้ชี้ไปยังไฟล์นั้น"
    ) from last_error


download_boundary(NATURAL_EARTH_URL, boundary_path)

# อ่านไฟล์ในเครื่อง (เสถียรกว่าการอ่านตรงจาก URL)
world = gpd.read_file(boundary_path)

# Natural Earth บางเวอร์ชันใช้ ADMIN บางเวอร์ชันใช้ NAME
if "ADMIN" in world.columns:
    thailand = world[world["ADMIN"] == "Thailand"].copy()
elif "NAME" in world.columns:
    thailand = world[world["NAME"] == "Thailand"].copy()
else:
    raise ValueError("ไม่พบ field ADMIN/NAME ใน Natural Earth dataset")

if len(thailand) == 0:
    raise ValueError("ไม่พบ polygon ของ Thailand ใน dataset")

# บังคับให้ CRS เป็น WGS84 ให้ตรงกับพิกัดสถานี
thailand = thailand.to_crs("EPSG:4326")

print("Thailand polygons:", len(thailand))
thailand

## 16. ดาวน์โหลดขอบเขตประเทศไทยจาก Natural Earth

ใช้ขอบเขตประเทศจริงสำหรับ cartographic context

> **หมายเหตุ (fix):** การอ่านไฟล์ตรงจาก URL ด้วย `gpd.read_file(URL)` มักล้มเหลวใน Colab
> เพราะ GDAL/pyogrio ใช้ CURL ภายในที่แพ้เรื่อง SSL/proxy/network ได้ง่าย
> เราจึง **ดาวน์โหลดด้วย `requests` ก่อน แล้วค่อยอ่านไฟล์ในเครื่อง** (รูปแบบเดียวกับตอนโหลด Air4Thai API)
> ไฟล์จะถูก cache ไว้ใน `DATA_DIR` เพื่อให้รันซ้ำได้เร็วและ reproducible

หาก network มีปัญหาจริง ๆ ผู้สอนสามารถอัปโหลด shapefile/GeoJSON ของประเทศไทยเองแล้วชี้ `boundary_path` มาที่ไฟล์นั้นได้

In [ ]:
# guard: ต้องมีสถานีที่มีพิกัดก่อนจึงจะพลอตได้
if len(station_gdf) == 0:
    raise ValueError("ไม่มีสถานีที่มีพิกัดให้พลอต ตรวจ QC ใน Cell ก่อนหน้า")

fig, ax = plt.subplots(figsize=(8.5, 11))

# Thailand boundary
thailand.plot(
    ax=ax,
    facecolor="0.95",
    edgecolor="0.25",
    linewidth=0.8
)

# Station points
station_gdf.plot(
    ax=ax,
    color="#d7301f",
    markersize=24,
    alpha=0.80,
    edgecolor="black",
    linewidth=0.35,
    label=f"Air4Thai stations (n = {len(station_gdf)})"
)

# Map extent — อิงขอบเขตประเทศ
minx, miny, maxx, maxy = thailand.total_bounds
pad_x = 0.6
pad_y = 0.5

ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_title(
    f"Air4Thai Air-Quality Monitoring Stations in Thailand\n"
    f"n = {len(station_gdf)} stations",
    fontsize=14
)
ax.legend(loc="lower left", frameon=True)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.5,
    alpha=0.4
)

# North arrow
ax.annotate(
    "N",
    xy=(0.94, 0.92),
    xytext=(0.94, 0.84),
    xycoords="axes fraction",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="-|>",
        linewidth=1.3
    )
)

plt.tight_layout()

static_png = FIG_DIR / "air4thai_stations_thailand.png"
static_pdf = FIG_DIR / "air4thai_stations_thailand.pdf"

plt.savefig(static_png, dpi=300, bbox_inches="tight")
plt.savefig(static_pdf, bbox_inches="tight")

plt.show()

print("Saved:")
print(static_png)
print(static_pdf)

## 18. Optional: พลอตแยกตาม Station Type

หาก API มี `stationType` หลายประเภท เราสามารถตรวจและแสดงด้วยสัญลักษณ์/กลุ่มต่างกันในงานขั้นต่อไป

In [ ]:
print("Station types:")
display(
    station_core["station_type"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("station_type")
    .reset_index(name="n")
)

## 19. Interactive Map ด้วย Folium

ข้อดี:

- zoom/pan ได้
- click station เพื่อดู metadata
- เหมาะสำหรับให้นิสิตสำรวจสถานี
- ใช้ `MarkerCluster` เพื่อไม่ให้จุดทับกันมากเมื่อดูระดับประเทศไทย

In [ ]:
center_lat = station_gdf["latitude"].mean()
center_lon = station_gdf["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles="OpenStreetMap",
    control_scale=True
)

cluster = MarkerCluster(
    name="Air4Thai stations"
).add_to(m)

for _, row in station_gdf.iterrows():

    station_code = str(row.get("station_code", ""))
    name_th = str(row.get("station_name_th", ""))
    name_en = str(row.get("station_name_en", ""))
    area_th = str(row.get("area_th", ""))
    station_type = str(row.get("station_type", ""))

    popup_html = f'''
    <div style="width:300px">
      <b>Station:</b> {station_code}<br>
      <b>Name (TH):</b> {name_th}<br>
      <b>Name (EN):</b> {name_en}<br>
      <b>Area:</b> {area_th}<br>
      <b>Type:</b> {station_type}<br>
      <b>Latitude:</b> {row["latitude"]:.6f}<br>
      <b>Longitude:</b> {row["longitude"]:.6f}
    </div>
    '''

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=5,
        weight=1,
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{station_code} | {name_th}",
        popup=folium.Popup(
            popup_html,
            max_width=350
        )
    ).add_to(cluster)

folium.LayerControl().add_to(m)

m

## 20. Save Interactive Map

In [ ]:
interactive_html = FIG_DIR / "air4thai_stations_thailand_interactive.html"

m.save(str(interactive_html))

print("Saved:", interactive_html)

## 21. Optional: Export GeoJSON สำหรับ QGIS / ArcGIS

In [ ]:
geojson_path = DATA_DIR / "air4thai_station_metadata.geojson"

station_gdf.to_file(
    geojson_path,
    driver="GeoJSON"
)

print("Saved:", geojson_path)

## 22. ตัวอย่างการ Join พิกัดเข้ากับข้อมูล PM₂.₅ รายวัน

สมมติ daily data มีคอลัมน์ `station_code`

In [ ]:
# ตัวอย่างเท่านั้น — แก้ path เป็นไฟล์จริงของคุณ
#
# daily_pm25 = pd.read_csv(
#     "/content/drive/MyDrive/.../pm2.5_daily_2025_single_sheet.csv"
# )
#
# daily_pm25["station_code"] = (
#     daily_pm25["station_code"]
#     .astype(str)
#     .str.strip()
#     .str.upper()
# )
#
# daily_with_coords = daily_pm25.merge(
#     station_core[
#         [
#             "station_code",
#             "latitude",
#             "longitude",
#             "station_type"
#         ]
#     ],
#     on="station_code",
#     how="left",
#     validate="many_to_one"
# )
#
# print(daily_with_coords.shape)
# display(daily_with_coords.head())

### ทำไมใช้ `validate="many_to_one"`?

Daily PM₂.₅ มีหลายวันต่อหนึ่งสถานี  
แต่ metadata ควรมี **หนึ่ง record ต่อหนึ่ง station_code**

ดังนั้น relationship คือ:

**many daily observations → one station metadata record**

หาก metadata มี station code ซ้ำ Pandas จะเตือน/error ซึ่งช่วยป้องกันการ join ผิดโดยไม่รู้ตัว

## 23. สรุป Output

เมื่อรัน Notebook จบ จะได้:

```text
03_air4thai_station_metadata/
│
├── data/
│   ├── air4thai_station_metadata_FULL.csv
│   ├── air4thai_station_metadata_CORE.csv
│   └── air4thai_station_metadata.geojson
│
└── figures/
    ├── air4thai_stations_thailand.png
    ├── air4thai_stations_thailand.pdf
    └── air4thai_stations_thailand_interactive.html
```

### ความหมาย
- `FULL.csv` — ทุก field ที่ได้จาก Air4Thai API ณ วันที่ดาวน์โหลด
- `CORE.csv` — station metadata สำหรับ join กับ time-series data
- `GeoJSON` — ใช้กับ QGIS/ArcGIS/Python GIS
- `PNG/PDF` — static Thailand station map
- `HTML` — interactive station map

# Exercise สำหรับนิสิต

1. หา station code ที่อยู่ใกล้จังหวัดของตนเอง
2. แสดง station name, latitude และ longitude
3. เปิด interactive map และสำรวจสถานี
4. เลือก 3–5 สถานีที่ต้องการใช้ทำวิจัย
5. อธิบายว่าทำไมจึงเลือกสถานีเหล่านั้น
6. ตรวจว่ามี missing coordinate หรือ station code ซ้ำหรือไม่
7. นำ `CORE.csv` ไป join กับข้อมูล PM₂.₅ ที่วิเคราะห์ใน Notebook ก่อนหน้า

## ขั้นต่อไป
Notebook ถัดไปสามารถใช้ station metadata นี้เพื่อทำ:

**Saraburi + Lopburi + Nakhon Nayok → province boundaries → stations inside provinces → surrounding stations within 25/50/100 km → research study-area map**